# Alpamayo 1.5: Navigation-Conditioned Trajectory Prediction

Alpamayo 1.5 introduces the ability to **condition trajectory predictions on natural language navigation instructions**. Given a driving scene and an instruction like *"Turn right in 30m"*, the model produces trajectories that follow the specified route intention.

This notebook demonstrates:
1. **Basic trajectory prediction** with chain-of-causation reasoning
2. **Navigation-conditioned sampling** -- how the same scene yields different trajectory distributions depending on the navigation instruction
3. **Manual control** -- how to integrate navigation conditioning into your own pipeline

We use data from the NVIDIA [PhysicalAI-AV Dataset](https://huggingface.co/datasets/nvidia/PhysicalAI-Autonomous-Vehicles).

In [ ]:
import importlib
import json
import mediapy as mp
import matplotlib.pyplot as plt
import numpy as np

import torch
from alpamayo1_5.models.alpamayo1_5 import Alpamayo1_5
import alpamayo1_5.load_custom_dataset as load_custom_dataset_module
load_custom_dataset_module = importlib.reload(load_custom_dataset_module)
load_custom_dataset = load_custom_dataset_module.load_custom_dataset
from alpamayo1_5 import helper, nav_utils
from alpamayo1_5.viz_utils import make_camera_grid, plot_bev_comparison

TRAJECTORY_DISPLAY_MODES = {"distribution", "heuristic", "mean", "median"}
trajectory_display_mode = "distribution"
plot_all_trajectory_samples = False


def select_prediction_tensor(pred_xyz, nav_text, display_mode=None):
    """Return predictions for the requested visualization mode.

    "distribution" preserves the notebook's original all-samples view.
    The other modes collapse samples to one displayed path. "heuristic"
    matches the batch exporter command-aware selection rule.
    """
    mode = trajectory_display_mode if display_mode is None else display_mode
    if mode not in TRAJECTORY_DISPLAY_MODES:
        raise ValueError(f"display_mode must be one of {sorted(TRAJECTORY_DISPLAY_MODES)}")
    if mode == "distribution":
        return pred_xyz

    pred_np = pred_xyz.detach().float().cpu().numpy()[0, 0]
    if pred_np.shape[0] == 0:
        return pred_xyz

    if mode == "mean":
        selected = pred_np.mean(axis=0)
    elif mode == "median":
        selected = np.median(pred_np, axis=0)
    else:
        nav_lower = nav_text.lower()
        final_lateral = pred_np[:, -1, 1]
        final_forward = pred_np[:, -1, 0]
        if "left" in nav_lower:
            sample_idx = int(np.argmax(final_lateral))
        elif "right" in nav_lower:
            sample_idx = int(np.argmin(final_lateral))
        elif "straight" in nav_lower:
            sample_idx = int(np.argmin(np.abs(final_lateral)))
        else:
            sample_idx = int(np.argmax(final_forward - np.abs(final_lateral)))
        selected = pred_np[sample_idx]

    selected = torch.tensor(selected, device=pred_xyz.device, dtype=pred_xyz.dtype)
    return selected.view(1, 1, 1, *selected.shape)


def _overlay_selected_path(ax, pred_xyz, nav_text, color, label, mode):
    selected = select_prediction_tensor(pred_xyz, nav_text, mode)
    xy = selected[0, 0, 0, :, :2].detach().float().cpu().numpy()
    ax.plot(
        xy[:, 0],
        xy[:, 1],
        color=color,
        linewidth=3.2,
        linestyle="--",
        label=f"selected {label} [{mode}]",
    )


def plot_nav_comparison(
    nav_result,
    data,
    camera_images,
    title,
    display_mode=None,
    plot_all_samples=None,
):
    mode = trajectory_display_mode if display_mode is None else display_mode
    show_all = plot_all_trajectory_samples if plot_all_samples is None else plot_all_samples
    use_distribution_view = mode == "distribution" or show_all
    if mode == "distribution":
        plot_title = title
    elif show_all:
        plot_title = f"{title} [{mode} + all samples]"
    else:
        plot_title = f"{title} [{mode}]"

    fig = plot_bev_comparison(
        pred_with_nav=(
            nav_result.pred_with_nav
            if use_distribution_view
            else select_prediction_tensor(nav_result.pred_with_nav, nav_result.nav_text, mode)
        ),
        pred_no_nav=(
            nav_result.pred_no_nav
            if use_distribution_view
            else select_prediction_tensor(nav_result.pred_no_nav, "", mode)
        ),
        pred_counterfactual=(
            nav_result.pred_counterfactual
            if use_distribution_view
            else select_prediction_tensor(
                nav_result.pred_counterfactual, nav_result.nav_text_swapped, mode
            )
        ),
        nav_text=nav_result.nav_text,
        nav_text_swapped=nav_result.nav_text_swapped,
        gt_future_xyz=data.get("ego_future_xyz"),
        camera_images=camera_images,
        title=plot_title,
    )

    if show_all and mode != "distribution":
        ax = fig.axes[-1]
        _overlay_selected_path(ax, nav_result.pred_with_nav, nav_result.nav_text, "tab:blue", "nav", mode)
        _overlay_selected_path(ax, nav_result.pred_no_nav, "", "tab:red", "no-nav", mode)
        _overlay_selected_path(
            ax,
            nav_result.pred_counterfactual,
            nav_result.nav_text_swapped,
            "tab:green",
            "opposite-nav",
            mode,
        )
        ax.legend(loc="upper left", fontsize=7)
        fig.tight_layout()

    return fig


## Part 1: Basic Trajectory Prediction

First, we load the model and run standard inference -- the model observes camera images and ego history, then generates a chain-of-causation (CoC) reasoning trace followed by a predicted trajectory.

In [ ]:
model = Alpamayo1_5.from_pretrained("nvidia/Alpamayo-1.5-10B", dtype=torch.bfloat16).to("cuda")
processor = helper.get_processor(model.tokenizer)

We load a scene at an intersection on campus where the vehicle approaches a decision point -- it could turn right or continue straight.

In [ ]:
segment_dir = "../../datasets/route_3/segment_01"
frame_idx = 60
exclude_cameras = [] # Include all cameras. Camera ids: 0 left, 1 wide, 2 right, 6 front
swap_front_cameras = False # Set True to swap camera ids 1 (front wide) and 6 (front)

def swap_camera_views(data, camera_a=1, camera_b=6):
    data = data.copy()
    camera_indices = data["camera_indices"]
    idx_a = torch.where(camera_indices == camera_a)[0]
    idx_b = torch.where(camera_indices == camera_b)[0]
    if len(idx_a) == 0 or len(idx_b) == 0:
        print(f"Could not swap cameras {camera_a} and {camera_b}; one of them is missing.")
        return data
    image_frames = data["image_frames"].clone()
    a = int(idx_a[0])
    b = int(idx_b[0])
    image_frames[[a, b]] = image_frames[[b, a]]
    data["image_frames"] = image_frames
    print(f"Swapped camera views for ids {camera_a} and {camera_b}.")
    return data

data_scene1 = load_custom_dataset(segment_dir, frame_idx, exclude_cameras=exclude_cameras)
if swap_front_cameras:
    data_scene1 = swap_camera_views(data_scene1, 1, 6)
mp.show_images(data_scene1["image_frames"].flatten(0, 1).permute(0, 2, 3, 1), columns=4, width=200)

In [ ]:
messages = helper.create_message(
    data_scene1["image_frames"].flatten(0, 1),
    camera_indices=data_scene1["camera_indices"],
)
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=False,
    continue_final_message=True,
    return_dict=True,
    return_tensors="pt",
)
model_inputs = helper.to_device(
    {
        "tokenized_data": inputs,
        "ego_history_xyz": data_scene1["ego_history_xyz"],
        "ego_history_rot": data_scene1["ego_history_rot"],
    },
    "cuda",
)

torch.cuda.manual_seed_all(42)
with torch.autocast("cuda", dtype=torch.bfloat16):
    pred_xyz, pred_rot, extra = model.sample_trajectories_from_data_with_vlm_rollout(
        data=model_inputs,
        top_p=0.98,
        temperature=0.6,
        num_traj_samples=1,
        max_generation_length=256,
        return_extra=True,
    )

print("Chain-of-Causation:\n", extra["cot"][0])


In [ ]:
for i in range(pred_xyz.shape[2]):
    pred_xy = pred_xyz.cpu()[0, 0, i, :, :2].T.numpy()
    gt_xy = data_scene1["ego_future_xyz"].cpu()[0, 0, :, :2].T.numpy()
    plt.plot(pred_xy[0], pred_xy[1], "o-", label=f"Predicted #{i + 1}")
plt.plot(gt_xy[0], gt_xy[1], "r-", label="Ground Truth")
plt.xlabel("x (m)")
plt.ylabel("y (m)")
plt.legend(loc="best")
plt.axis("equal")
plt.title("Basic Trajectory Prediction (no navigation instruction)")
plt.show()


## Part 2: Navigation-Conditioned Trajectory Sampling

One of the key new capabilities of Alpamayo 1.5: by providing a **navigation instruction**, you can steer the model's trajectory distribution.

We demonstrate this with a three-condition comparison:
- **p(traj | nav)** (blue): conditioned on the actual navigation instruction
- **p(traj)** (red): no navigation instruction (baseline)
- **p(traj | opposite nav)** (green): conditioned on the *opposite* direction (counterfactual)
- **GT** (black): ground truth trajectory

The same model and inference API are used for all three -- only the input text prompt changes.

Note: You can use diffusion temperature < 1.0 to get more stable but less diversity trajectory samples.
Set `trajectory_display_mode` to `"distribution"`, `"heuristic"`, `"mean"`, or `"median"`. The default `"distribution"` keeps the original notebook view with all samples, a bold median path, and endpoint density. For selected modes, set `plot_all_trajectory_samples = True` to show every sampled path and overlay the selected representative path.


In [ ]:
# Choose how trajectory samples are displayed.
# Options: "distribution", "heuristic", "mean", "median"
trajectory_display_mode = "distribution"
plot_all_trajectory_samples = False

nav_text = "Turn left in 15m"

print(f"Navigation instruction: {nav_text}")
print(f"Counterfactual (swapped): {nav_utils.swap_direction(nav_text)}")
print(f"Trajectory display mode: {trajectory_display_mode}")
print(f"Plot all trajectory samples: {plot_all_trajectory_samples}")


In [ ]:
torch.cuda.manual_seed_all(42)
with torch.autocast("cuda", dtype=torch.bfloat16):
    nav_result = nav_utils.compare_nav_conditions(
        model=model,
        processor=processor,
        data=data_scene1,
        nav_text=nav_text,
        num_traj_samples=16,
        top_p=0.98,
        temperature=0.6,
        max_generation_length=256,
        return_extra=True,
        additional_nav_inference_kwargs={
            "diffusion_kwargs": {
                # The temperature for controlling the initial noise. Note that using
                # temperature < 1.0 will result in a more stable sampling with less diversity.
                "temperature": 0.6,
            }
        },
    )

print(f"Trajectories per condition: {nav_result.pred_with_nav.shape[2]}")

In [ ]:
camera_grid = make_camera_grid(
    data_scene1["image_frames"], camera_indices=data_scene1["camera_indices"]
)

fig = plot_nav_comparison(
    nav_result=nav_result,
    data=data_scene1,
    camera_images=camera_grid,
    title=f'Navigation: "{nav_text}"',
)
plt.show()


### Classifier-Free Guidance (CFG) for Navigation-Conditioned Inference

Optionally, we can apply **classifier-free guidance** ([Ho & Salimans, 2022](https://arxiv.org/abs/2207.12598)) to amplify the effect of the navigation instruction on trajectory prediction. The `inference_guidance_weight` parameter ($\alpha$) relates to $w$ in Eq. (6) of the paper by $\alpha = 1 + w$:

- **$\alpha = 1$**: no guidance amplification (default conditioned prediction).
- **$\alpha > 1$**: amplified guidance -- the model more aggressively follows the navigation instruction at the expense of sample diversity.
- **$\alpha = 0$**: pure unconditioned prediction (ignores the navigation instruction entirely).

NOTE: To run this inference with CFG, you will need 60GB+ GPU memory.

In [ ]:
torch.cuda.manual_seed_all(42)
with torch.autocast("cuda", dtype=torch.bfloat16):
    nav_result = nav_utils.compare_nav_conditions(
        model=model,
        processor=processor,
        data=data_scene1,
        nav_text=nav_text,
        num_traj_samples=16,
        top_p=0.98,
        temperature=0.6,
        max_generation_length=256,
        return_extra=False,
        # comment out the following lines to use the default inference function
        nav_inference_fn=model.sample_trajectories_from_data_with_vlm_rollout_cfg_nav,
        additional_nav_inference_kwargs={
            "diffusion_kwargs": {
                "use_classifier_free_guidance": True,
                # 0 = unguided only, 1 = guidance only, >1 = more guidance
                "inference_guidance_weight": 1.5,
                # The temperature for controlling the initial noise. Note that using
                # temperature < 1.0 will result in a more stable sampling with less diversity.
                "temperature": 0.6,
            }
        },
    )

print(f"Trajectories per condition: {nav_result.pred_with_nav.shape[2]}")

In [ ]:
camera_grid = make_camera_grid(
    data_scene1["image_frames"], camera_indices=data_scene1["camera_indices"]
)

fig = plot_nav_comparison(
    nav_result=nav_result,
    data=data_scene1,
    camera_images=camera_grid,
    title=f'Navigation: "{nav_text}"',
)
plt.show()


### Try different instructions

Use this local list as a menu of example navigation instructions. Copy any of these into the `nav_texts` list in Part 3 to compare multiple commands.

In [ ]:
nav_instruction_examples = [
    "Continue straight through the intersection",
    "Go straight for 50m. Do not turn left or right.",
    "Stay in the current lane and continue straight.",
    "Turn left in 10m",
    "Turn right in 15m",
]

print("Example navigation instructions:")
for i, instruction in enumerate(nav_instruction_examples, start=1):
    print(f"{i}. {instruction}")

## Part 3: Compare Multiple Navigation Commands

Use `nav_texts` to run several navigation commands on the same scene. For each command, the notebook compares nav-conditioned, no-nav, and opposite-nav predictions.

### Run multi-command comparisons

Edit the `nav_texts` list below to try different maneuvers or distances. The frame, segment, and camera settings are reused from the scene loaded at the beginning of the notebook.

In [ ]:
nav_texts = [
    "Turn left in 5m",
    "Turn left in 10m",
    "Turn left in 15m",
    "Turn left in 20m",
]

print(f"Trajectory display mode: {trajectory_display_mode}")
print(f"Plot all trajectory samples: {plot_all_trajectory_samples}")

camera_grid_scene1 = make_camera_grid(
    data_scene1["image_frames"], camera_indices=data_scene1["camera_indices"]
)

for nav_text in nav_texts:
    print(f"Navigation instruction: {nav_text}")
    print(f"Counterfactual (swapped): {nav_utils.swap_direction(nav_text)}")

    torch.cuda.manual_seed_all(42)
    with torch.autocast("cuda", dtype=torch.bfloat16):
        nav_result = nav_utils.compare_nav_conditions(
            model=model,
            processor=processor,
            data=data_scene1,
            nav_text=nav_text,
            num_traj_samples=16,
            top_p=0.98,
            temperature=0.6,
            max_generation_length=256,
            return_extra=True,
            additional_nav_inference_kwargs={
                "diffusion_kwargs": {
                    "temperature": 0.6,
                }
            },
        )

    print(f"Trajectories per condition: {nav_result.pred_with_nav.shape[2]}")
    if nav_result.extra_with_nav is not None:
        print(f"Chain-of-Causation (nav-conditioned):\n{nav_result.extra_with_nav['cot'][0]}")

    fig = plot_nav_comparison(
        nav_result=nav_result,
        data=data_scene1,
        camera_images=camera_grid_scene1,
        title=f'Navigation: "{nav_text}"',
    )
    ax = fig.axes[-1]
    x_max = max(
        float(nav_result.pred_with_nav[..., 0].max()),
        float(nav_result.pred_no_nav[..., 0].max()),
        float(nav_result.pred_counterfactual[..., 0].max()),
    )
    if data_scene1.get("ego_future_xyz") is not None:
        x_max = max(x_max, float(data_scene1["ego_future_xyz"][..., 0].max().cpu()))
    ax.set_xlim(0, max(30, x_max + 5))
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), borderaxespad=0, fontsize=7)
    fig.tight_layout(rect=[0, 0, 0.78, 1])
    plt.show()
    print("-" * 80)
